# 05 · Image-model check (RQ4)

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 29 Oct 2026 · **Needs:** `results/01_inventory/` to `results/04_posthoc/` · **Plan:** section **7d**
(approved 29 Oct 2026, fixed before any image was downloaded)

**What this notebook does, in plain words**

Notebooks 03 and 04 showed that for **four tasks** the scan's "label card" (machine, settings, description words) alone
predicts the diagnosis class. This notebook asks: **does a real image model pick up the same thing?**
1. **Images:** it downloads the same main CT series the audit used, for all four flagged tasks (about 50 GB, straight
   from TCIA into Colab's own disk, not into Drive).
2. **The image model, "the usual way":** two public CT foundation models (CT-FM and Merlin) turn each scan into a list of
   numbers (an *embedding*); the same simple logistic regression as notebook 03 learns the class from it.
3. **Acquisition-controlled checks:** does the image score still separate the classes *inside* one hospital (or one
   description language, or one scanner vendor)? With whole hospitals held out? Does it add anything to the label card?
4. **Counterfactual checks (biology removed):**
   - **Background only:** the whole body is replaced by air. Only the table, the air and the image noise are left. If
     the class can still be predicted, the images carry acquisition fingerprints.
   - **Acquisition swap:** thin-slice scans are made thick, like the other class. If predictions move toward the other
     class (more than with an equally strong control change), the model reacts to slice thickness.
5. **DICOM headers:** the downloaded files have fields the TCIA API lacks (true slice thickness, kernel, kVp, ...). The
   notebook 03 model is rerun with them.

**Rehearsal first.** With the default `RUN_MODE = "both"`, the analysis first runs on **shuffled labels** (a rehearsal:
every number should look like chance), checks itself, and only then runs on the real labels.

**How to run:** in Colab choose *Runtime → Change runtime type → L4 GPU (or A100), High-RAM*, then *Runtime → Run all*.
Sign in with **the Google account that holds the project Drive folder** when Drive asks. Expect about 3–5 hours; keep the tab open (Colab Pro+ background
execution also works). **If Colab disconnects, run all again:** finished scans and finished tasks are loaded, not redone.
The GPU is needed only for the two encoders; most of the time goes to downloading and preparing the 525 scans.
Outputs go to `results/05_image_check/`.

In [ ]:
# 1. Setup: Drive, folders, GPU
import os, re, sys, json, time, copy, shutil, zipfile, hashlib, datetime, platform, warnings, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
    WORK = Path('/content/asa05_work')          # Colab's own disk: DICOM and NIfTI files, never Drive
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    WORK = Path(os.environ['ASA_WORK_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')
assert RUN_MODE in ('both', 'rehearsal', 'real'), RUN_MODE

IN01 = PROJECT / 'results' / '01_inventory'
IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
IN04 = PROJECT / 'results' / '04_posthoc'
OUT_ROOT = PROJECT / 'results' / '05_image_check'
CACHE = OUT_ROOT / 'cache'                       # label-free: headers, embeddings, QC pictures
for d in (IN01, IN02, IN03, IN04):
    assert (d / 'run_manifest.json').exists(), f'{d} is missing: run the earlier notebooks first.'
for d in (CACHE / 'headers', CACHE / 'embeddings', CACHE / 'qc', WORK / 'nifti', WORK / 'dicom'):
    d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
free_gb = shutil.disk_usage(WORK).free / 1e9
print('Project folder:', PROJECT)
print('Outputs to    :', OUT_ROOT)
print('Work disk     :', WORK, f'({free_gb:.0f} GB free)')
print('Run mode      :', RUN_MODE)
print('Run started   :', RUN_STARTED, '(UTC)')
if free_gb < 80:
    print('WARNING: less than 80 GB free on the work disk. The scans need about 35 GB there. '
          'Use an L4 or A100 runtime (they have more disk).')

In [ ]:
# 2. Packages (exact versions; installed into Colab only)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai==1.6.1', 'SimpleITK==2.5.6',
                    'pydicom==3.0.2', 'nibabel==5.4.2', 'merlin-vlm==0.0.7'], check=True)

import numpy as np
import pandas as pd
import requests
import scipy
from scipy import ndimage, sparse
from scipy.stats import rankdata, spearmanr
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed
import pydicom
import SimpleITK as sitk
import nibabel
import torch
import torchvision
import monai
from monai.transforms import (Compose, LoadImage, EnsureType, Orientation, Spacing, ScaleIntensityRange, SpatialCrop)
from monai.transforms.utils import generate_spatial_bounding_box
from monai.inferers import SlidingWindowSplitter
from monai.networks.nets.segresnet_ds import SegResEncoder
import huggingface_hub
from huggingface_hub import hf_hub_download
import safetensors.torch
import merlin
from merlin.models import i3res
from merlin.data.monai_transforms import ImageTransforms as MERLIN_TRANSFORMS
import importlib.metadata

warnings.filterwarnings('ignore', message='.*pixdim.*')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1
N_CPU = os.cpu_count()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, default=_jsonable))
    tmp.replace(path)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__,
            'torch': torch.__version__, 'torchvision': torchvision.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version.VersionString(), 'pydicom': pydicom.__version__,
            'nibabel': nibabel.__version__, 'huggingface_hub': huggingface_hub.__version__,
            'merlin-vlm': importlib.metadata.version('merlin-vlm'), 'cpus': N_CPU,
            'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'}
print(VERSIONS)
if IN_COLAB and DEVICE != 'cuda':
    print('WARNING: no GPU. Choose Runtime → Change runtime type → L4 GPU, then run all again.')

## Settings (plan section 7d)
The model settings are notebook 03's frozen settings (checked against its run record below). The image settings are
plan 7d and the details settled while building this notebook (plan 7d, "Settled while building").

In [ ]:
# 3. Settings
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part',
               'hdr_kernel', 'hdr_filter_type', 'hdr_patient_position'}
LOG_NUMERIC = {'image_count', 'body_area_cm2'}
TEXT = {'protocol_text', 'care_text'}

# The four tasks flagged by notebook 03 (and again by notebook 04's rerun), with the grouping for the within-group check
TASKS = {'renal subtype (KIRC vs KIRP)': 'hospital',
         'CPTAC-PDA | stage': 'description language',
         'lung histology (LUAD vs LUSC)': 'hospital',
         'LIDC-IDRI | primary vs metastatic': 'scanner vendor'}

# Images
TCIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
PREP_JOBS = max(1, min(8, N_CPU - 1))      # scans prepared in parallel
EMB_WORKERS = max(1, min(6, N_CPU - 2)) if IN_COLAB else 0   # CPU workers that preprocess for the GPU
BODY_HU = -500          # body mask: voxels above -500 HU ...
OPEN_MM = 3.0           # ... opened in-plane (detaches the table), largest 3D component, holes filled per slice ...
DILATE_MM = 5.0         # ... and dilated by 5 mm
AIR_HU = -1000          # background-only: every body voxel is set to air
N_QC = 2                # QC pictures for the first 2 patients (by ID) of each task

# Encoder 1: CT-FM (Pai et al. 2025, Apache-2.0). Weights from the Hugging Face model card; preprocessing and patch
# embedding exactly as the authors' own embedding script (github.com/project-lighter/CT-FM,
# scripts/3d_mir_generate_embeddings.py): SPL orientation, 3 x 1 x 1 mm, foreground crop (HU > 0), HU -1024..2048 -> 0..1,
# 24 x 128 x 128 patches with 0.625 overlap, last-level features average-pooled per patch (512), mean over patches.
CTFM_REPO, CTFM_REVISION = 'project-lighter/ct_fm_feature_extractor', '527b0d1c262b433396c85ff2de84c427b26945a1'
CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP, CTFM_BATCH = (3.0, 1.0, 1.0), (24, 128, 128), 0.625, 32
# Encoder 2: Merlin (Blankemeier et al., MIT license). Official checkpoint and the package's own image transforms
# (RAS, 1.5 x 1.5 x 3 mm, HU -1000..1000 -> 0..1, pad/crop 224 x 224 x 160); image embedding = 2048 pooled features.
MERLIN_REPO, MERLIN_REVISION = 'stanfordmimi/Merlin', '66dd07585de79e4d8d70d67806da88f3ccaa5a0c'
MERLIN_CKPT = 'i3_resnet_clinical_longformer_best_clip_04-02-2024_23-21-36_epoch_99.pt'
ENCODERS = ('ctfm', 'merlin')
LABEL_FREE_VARIANTS = ('original', 'background')

# Acquisition swap (thin -> thick; plan 7d, settled while building)
SWAP_MIN_DIFF_MM = 0.5      # run only if the class medians of slice spacing differ by at least this much
SWAP_MIN_GAP_MM = 0.1       # a thin-class scan is swapped only if its own spacing is below the target by more than this
SWAP_SHIFT_MIN = 0.05       # rule: mean shift toward the thick class >= 0.05, CI excludes 0, and above the control
CONTROL_SIGMAS_MM = (0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.5, 4.0, 5.0, 6.0)  # matched per patient

# DICOM-header audit (secondary, RQ2): technical fields only. Never: site, station, device serial, demographics, dates.
# Contrast is a clinical choice (plan 7c) and is reported, not used.
HDR_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_pixel_spacing', 'hdr_recon_diameter', 'hdr_kvp',
               'hdr_tube_current', 'hdr_exposure', 'hdr_ctdivol', 'hdr_pitch']
HDR_CATEGORICAL = ['hdr_kernel', 'hdr_filter_type', 'hdr_patient_position']
HDR_COLS = HDR_NUMERIC + HDR_CATEGORICAL

# Rehearsal gate (shuffled labels): every analysis ran, null means show no leakage, enough scans. The upper bound
# catches leakage; the lower bound is loose because cross-validated AUC under no signal sits below 0.5 (7b.11), more so
# with 512-2,048 embedding features than with the metadata.
GATE_NULL_RANGE = (0.35, 0.55)
GATE_MIN_COVERAGE = 0.95

PREP_KEY = hashlib.md5(json.dumps({'BODY_HU': BODY_HU, 'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU,
                                   'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY = hashlib.md5(json.dumps({'prep': PREP_KEY, 'ctfm': [CTFM_REVISION, CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP],
                                  'merlin': [MERLIN_REVISION, MERLIN_CKPT], 'v': 1},
                                 sort_keys=True).encode()).hexdigest()[:12]

# Local code tests only (ignored in Colab)
TEST = {}
if not IN_COLAB:
    TEST = {'patients': [p for p in os.environ.get('ASA_TEST_PATIENTS', '').split(';') if p],
            'fake_images': os.environ.get('ASA_TEST_FAKE_IMAGES') == '1',
            'stop_after_images': os.environ.get('ASA_TEST_STOP_AFTER_IMAGES') == '1',
            'swap_target_mm': float(os.environ['ASA_TEST_SWAP_TARGET']) if os.environ.get('ASA_TEST_SWAP_TARGET') else None,
            'skip_official_check': os.environ.get('ASA_TEST_SKIP_OFFICIAL') == '1',
            'real_is_shuffled': True}   # local tests never see the real labels
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    CTFM_BATCH = int(os.environ.get('ASA_TEST_CTFM_BATCH', CTFM_BATCH))   # laptop memory; results do not depend on it
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [ ]:
# 4. Load and verify the inputs from notebooks 01-04
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder.name} changed since that run: {bad}'
    return man

man01, man02, man03, man04 = verify(IN01), verify(IN02), verify(IN03), verify(IN04)
for name, man in (('03', man03), ('04', man04)):
    assert man['in_colab'] and not man['local_test_settings'], f'results of notebook {name} are not the real Colab run'
assert man03['input_notebook02_run'] == man02['run_started_utc'], 'notebook 03 was run on other notebook 02 outputs'
assert man04['input_notebook03_run'] == man03['run_started_utc'], 'notebook 04 was run on other notebook 03 outputs'
assert man04['reproduced_notebook03'], 'notebook 04 did not reproduce notebook 03'
s03 = man03['settings']
assert (s03['SEED'], s03['N_SPLITS'], s03['N_REPEATS'], s03['MIN_COUNT'], s03['LR_PARAMS']) == \
       (SEED, N_SPLITS, N_REPEATS, MIN_COUNT, LR_PARAMS), 'model settings differ from notebook 03'
v03 = man03['versions']
same_versions = (v03['numpy'], v03['pandas'], v03['scikit-learn']) == (np.__version__, pd.__version__, sklearn.__version__)
print('Inputs verified: notebooks 01-04 (03 run of', man03['run_started_utc'], ')')
print('Package versions', 'match notebook 03' if same_versions else
      f"DIFFER from notebook 03 (then numpy {v03['numpy']}, pandas {v03['pandas']}, scikit-learn {v03['scikit-learn']})")

a03 = pd.read_csv(IN03 / 'audit_primary.csv')
b04 = pd.read_csv(IN04 / 'posthoc_blocklist_rerun.csv')
assert sorted(a03[a03.flagged].task_id) == sorted(TASKS), 'the flagged tasks of notebook 03 differ from plan 7d'
assert sorted(b04[b04.flagged].task_id) == sorted(TASKS), 'the flagged tasks of notebook 04 differ from plan 7d'
oof03 = pd.read_csv(IN03 / 'oof_predictions.csv.gz')

labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
main = pd.read_csv(IN01 / 'main_series.csv', low_memory=False)
smeta = pd.read_csv(IN01 / 'series_metadata.csv.gz', usecols=['SeriesInstanceUID', 'FileSize']).drop_duplicates(
    'SeriesInstanceUID')
KEY = ['collection', 'PatientID', 'modality']

# notebook 03's patient features (original word list)
text = (toks0.assign(group=np.where(toks0.category == 'care pathway', 'care_text', 'protocol_text'))
             .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat0.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
POLISH_WORDS = set(man04['word_lists']['POLISH_WORDS'])
polish = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)

# the patients and their main series (rule 7a.7, frozen in notebook 01)
lab = labels[labels.task_id.isin(TASKS)]
pts = lab.merge(main[KEY + ['SeriesInstanceUID', 'ImageCount']], on=KEY, how='left', validate='one_to_one')
pts = pts.merge(smeta, on='SeriesInstanceUID', how='left')
assert pts.SeriesInstanceUID.notna().all(), 'a task patient has no main series'
pts['key'] = pts.collection + '__' + pts.PatientID
PATIENTS = pts.drop_duplicates('key').sort_values('key').reset_index(drop=True)
if TEST.get('patients'):
    PATIENTS = PATIENTS[PATIENTS.key.isin(TEST['patients'])].reset_index(drop=True)
QC_KEYS = set(pts[pts.key.isin(PATIENTS.key)].sort_values('key').groupby('task_id').key.head(N_QC))
print(f"{len(PATIENTS)} patients, {PATIENTS.FileSize.sum() / 1e9:.1f} GB of DICOM to download")
show(pts.groupby('task_id').agg(patients=('key', 'size'), GB=('FileSize', lambda s: round(s.sum() / 1e9, 1))))

## Part 1 · Download and prepare every scan (no labels used)
For each patient: download the main series from TCIA, read it in order, save it as a NIfTI volume (in HU), read the
DICOM header fields, make the body mask and the background-only version. Everything label-free is cached in Drive
(`cache/`), so a second run (or the real-label pass) does not repeat it.

In [ ]:
# 5. Download, read and convert one series
def download_series(uid, dest):
    last = None
    for wait in (0, 10, 30, 90, 240):
        time.sleep(wait)
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(f'{TCIA}/getImage', params={'SeriesInstanceUID': uid}, stream=True,
                              timeout=(30, 600)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(tmp) as z:
                badf = z.testzip()
                assert badf is None, f'corrupt file in zip: {badf}'
            tmp.replace(dest)
            return dest.stat().st_size
        except Exception as e:   # network errors, partial files: try again
            last = e
    raise RuntimeError(f'download failed 5 times: {last!r}')

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def _text(v):
    if v is None:
        return np.nan
    if isinstance(v, pydicom.multival.MultiValue):
        v = '/'.join(str(x) for x in v)
    v = str(v).strip().upper()
    return v if v else np.nan

def read_series(folder, uid):
    """The slices of one series, sorted along the slice normal, with geometry checks."""
    metas = []
    for p in sorted(folder.rglob('*')):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
        except Exception:
            continue
        if str(getattr(ds, 'SeriesInstanceUID', '')) != uid:
            continue
        if not all(t in ds for t in ('ImagePositionPatient', 'ImageOrientationPatient', 'PixelSpacing', 'Rows')):
            continue
        metas.append((p, ds))
    assert metas, 'no image slices in the download'
    combo = [(tuple(np.round(np.asarray(ds.ImageOrientationPatient, float), 3)), int(ds.Rows), int(ds.Columns))
             for _, ds in metas]
    major = pd.Series(combo).value_counts().index[0]
    keep = [m for m, c in zip(metas, combo) if c == major]
    iop = np.asarray(keep[0][1].ImageOrientationPatient, float)
    row_cos, col_cos = iop[:3], iop[3:]
    normal = np.cross(row_cos, col_cos)
    pos = np.array([[float(v) for v in ds.ImagePositionPatient] for _, ds in keep])
    proj = pos @ normal
    inst = np.array([_num(getattr(ds, 'InstanceNumber', np.nan)) for _, ds in keep])
    order = np.lexsort((np.nan_to_num(inst, nan=1e9), proj))
    dup = np.r_[False, np.diff(proj[order]) < 1e-3]          # same position twice: keep the first
    order = order[~dup]
    keep, pos, proj = [keep[i] for i in order], pos[order], proj[order]
    assert len(keep) >= 10, f'only {len(keep)} usable slices'
    gaps = np.diff(proj)
    z = float(np.median(gaps))
    step = pos[-1] - pos[0]
    geo = {'row_cos': row_cos, 'col_cos': col_cos, 'normal': normal, 'origin': pos[0], 'z': z,
           'n_other_orientation_dropped': sum(c != major for c in combo),
           'n_duplicates_dropped': int(dup.sum()),
           'max_gap_deviation_mm': float(np.max(np.abs(gaps - z))),
           'nonuniform_spacing': bool(np.max(np.abs(gaps - z)) > max(0.05, 0.1 * z)),
           'gantry_tilt_deg': float(np.degrees(np.arccos(min(1.0, abs(step @ normal) / np.linalg.norm(step))))),
           'coverage_mm': float(proj[-1] - proj[0] + z)}
    return keep, geo

def header_fields(keep, geo):
    ds0 = keep[0][1]
    def med(name):
        vals = [_num(getattr(ds, name, None)) for _, ds in keep]
        vals = [v for v in vals if np.isfinite(v)]
        return float(np.median(vals)) if vals else np.nan
    ps = [float(v) for v in ds0.PixelSpacing]
    agent = getattr(ds0, 'ContrastBolusAgent', None)
    return {'hdr_manufacturer': _text(getattr(ds0, 'Manufacturer', None)),
            'hdr_model': _text(getattr(ds0, 'ManufacturerModelName', None)),
            'hdr_slice_thickness': med('SliceThickness'), 'hdr_z_spacing': geo['z'],
            'hdr_pixel_spacing': float(np.mean(ps)), 'hdr_rows': int(ds0.Rows), 'hdr_columns': int(ds0.Columns),
            'hdr_n_slices': len(keep), 'hdr_coverage_mm': geo['coverage_mm'],
            'hdr_recon_diameter': med('ReconstructionDiameter'), 'hdr_kvp': med('KVP'),
            'hdr_tube_current': med('XRayTubeCurrent'), 'hdr_exposure': med('Exposure'),
            'hdr_ctdivol': med('CTDIvol'), 'hdr_pitch': med('SpiralPitchFactor'),
            'hdr_kernel': _text(getattr(ds0, 'ConvolutionKernel', None)),
            'hdr_filter_type': _text(getattr(ds0, 'FilterType', None)),
            'hdr_patient_position': _text(getattr(ds0, 'PatientPosition', None)),
            'hdr_contrast_agent_given': ('yes' if agent is not None and str(agent).strip() else 'no'),
            'hdr_gantry_tilt_deg': geo['gantry_tilt_deg'], 'hdr_nonuniform_spacing': geo['nonuniform_spacing'],
            'hdr_max_gap_deviation_mm': geo['max_gap_deviation_mm'],
            'hdr_duplicates_dropped': geo['n_duplicates_dropped'],
            'hdr_other_orientation_dropped': geo['n_other_orientation_dropped']}

def to_image(keep, geo):
    """HU volume with the geometry from the DICOM positions (not from the reader's guess)."""
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(p) for p, _ in keep])
    reader.SetOutputPixelType(sitk.sitkFloat32)          # GDCM applies the rescale slope and intercept (HU)
    img = reader.Execute()
    ps = [float(v) for v in keep[0][1].PixelSpacing]      # [between rows, between columns]
    img.SetSpacing((ps[1], ps[0], geo['z']))
    img.SetOrigin(tuple(float(v) for v in geo['origin']))
    img.SetDirection(tuple(np.stack([geo['row_cos'], geo['col_cos'], geo['normal']], axis=1).ravel()))
    return img

def like(arr, ref, dtype=np.int16):
    """A SimpleITK image from a (z, y, x) array, with the geometry of `ref`."""
    if dtype == np.int16:
        arr = np.clip(np.round(arr), -32768, 32767)
    out = sitk.GetImageFromArray(arr.astype(dtype))
    out.CopyInformation(ref)
    return out

In [ ]:
# 6. Body mask, background-only version, QC picture
def body_mask(arr, sp):
    """arr: (z, y, x) HU; sp: (sz, sy, sx) mm. Returns (body, body dilated by DILATE_MM)."""
    m = arr > BODY_HU
    r = max(1, int(round(OPEN_MM / ((sp[1] + sp[2]) / 2))))
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    m = ndimage.binary_opening(m, structure=(yy ** 2 + xx ** 2 <= r * r)[None])
    lab_, n = ndimage.label(m)
    if n == 0:
        return np.zeros_like(m), np.zeros_like(m)
    counts = np.bincount(lab_.ravel())
    counts[0] = 0
    largest = lab_ == counts.argmax()
    del lab_, m
    core = ndimage.binary_fill_holes(largest, structure=np.ones((1, 3, 3), bool))
    del largest
    # dilation by DILATE_MM, computed inside the bounding box plus a margin (same result, much faster)
    pad = [int(np.ceil(DILATE_MM / s)) + 1 for s in sp]
    idx = np.where(core)
    lo = [max(0, int(i.min()) - p) for i, p in zip(idx, pad)]
    hi = [min(n_, int(i.max()) + p + 1) for i, n_, p in zip(idx, core.shape, pad)]
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    dil = np.zeros_like(core)
    dil[box] = ndimage.distance_transform_edt(~core[box], sampling=sp).astype(np.float32) <= DILATE_MM
    return core, dil

def qc_picture(key, arr, core, bg, sp, path):
    from matplotlib.figure import Figure                 # no pyplot: works the same in worker processes
    z = int(np.argmax(core.sum(axis=(1, 2))))          # the slice with the most body
    fig = Figure(figsize=(12, 4.2))
    ax = fig.subplots(1, 3)
    kw = dict(cmap='gray', vmin=-1000, vmax=400)
    ax[0].imshow(arr[z], **kw); ax[0].contour(core[z], levels=[0.5], colors='r', linewidths=0.6)
    ax[0].set_title('original + body outline')
    ax[1].imshow(bg[z], **kw); ax[1].set_title('background only')
    ax[2].imshow(arr[::-1, arr.shape[1] // 2, :], aspect=sp[0] / sp[2], **kw); ax[2].set_title('coronal (original)')
    for a in ax:
        a.axis('off')
    fig.suptitle(key, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=80)

def nifti_paths(key):
    d = WORK / 'nifti'
    return {'original': d / f'{key}__original.nii.gz', 'background': d / f'{key}__background.nii.gz',
            'mask': d / f'{key}__mask.nii.gz'}

def header_path(key):
    return CACHE / 'headers' / f'{key}.json'

def header_ok(key):
    p = header_path(key)
    return p.exists() and json.loads(p.read_text()).get('prep_key') == PREP_KEY

In [ ]:
# 7. Prepare one patient (runs in parallel workers)
def prepare_patient(key, uid, file_size):
    paths = nifti_paths(key)
    if header_ok(key) and all(p.exists() for p in paths.values()):
        return {'key': key, 'status': 'ready'}
    t0 = time.time()
    tmp = WORK / 'dicom' / key
    zpath = WORK / 'dicom' / f'{key}.zip'
    try:
        shutil.rmtree(tmp, ignore_errors=True)
        nbytes = download_series(uid, zpath)
        with zipfile.ZipFile(zpath) as z:
            n_files = len([n for n in z.namelist() if not n.endswith('/')])
            z.extractall(tmp)
        keep, geo = read_series(tmp, uid)
        hdr = header_fields(keep, geo)
        img = to_image(keep, geo)
        arr = sitk.GetArrayFromImage(img)                  # (z, y, x), float32 HU
        sp = img.GetSpacing()[::-1]
        core, dil = body_mask(arr, sp)
        bg = arr.copy()
        bg[dil] = AIR_HU
        for name, a, dt in (('original', arr, np.int16), ('background', bg, np.int16), ('mask', dil, np.uint8)):
            sitk.WriteImage(like(a, img, dt), str(paths[name]), useCompression=True)
        area = core.sum(axis=(1, 2)) * sp[1] * sp[2] / 100.0
        sub = arr[::2, ::4, ::4]                           # a regular subsample is enough for the QC percentiles
        rec = {'key': key, 'SeriesInstanceUID': uid, 'prep_key': PREP_KEY, 'download_bytes': nbytes,
               'tcia_file_size': file_size, 'n_files_in_zip': n_files, **hdr,
               'body_area_cm2': float(area[area > 0].mean()) if (area > 0).any() else np.nan,
               'body_fraction_dilated': float(dil.mean()),
               'hu_p01': float(np.percentile(sub, 1)), 'hu_p50': float(np.percentile(sub, 50)),
               'hu_p99': float(np.percentile(sub, 99)), 'prep_seconds': round(time.time() - t0, 1)}
        if key in QC_KEYS:
            qc_picture(key, arr, core, bg, sp, CACHE / 'qc' / f'{key}.png')
        write_json_atomic(header_path(key), rec)
        return {'key': key, 'status': 'ok', 'seconds': rec['prep_seconds']}
    except Exception as e:
        return {'key': key, 'status': 'failed', 'error': repr(e)[:400]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
        zpath.unlink(missing_ok=True)

### Encoders
Both encoders are loaded from pinned revisions. Merlin's image encoder is built without its text half and without the
ImageNet download; every weight comes from the official checkpoint (strict loading), so its image embedding is the same
as the official `Merlin(ImageEmbedding=True)`. A comparison with the official class is run once and recorded.
Gradient checkpointing (a memory saver for training) is switched off in Merlin's code; the output does not change.

In [ ]:
# 8. Load the two encoders
i3res.checkpoint = type('NoCheckpoint', (), {'checkpoint': staticmethod(lambda fn, *a, **k: fn(*a))})

def load_ctfm():
    cfg = json.loads(Path(hf_hub_download(CTFM_REPO, 'config.json', revision=CTFM_REVISION)).read_text())
    model = SegResEncoder(**cfg)
    state = safetensors.torch.load_file(hf_hub_download(CTFM_REPO, 'model.safetensors', revision=CTFM_REVISION))
    model.load_state_dict(state, strict=True)
    return model.eval()

MERLIN_DIR = Path(merlin.__file__).parent / 'models' / 'checkpoints'   # where the official class looks for it
if not IN_COLAB and os.environ.get('ASA_TEST_MERLIN_DIR'):              # local tests: Windows path-length limit
    MERLIN_DIR = Path(os.environ['ASA_TEST_MERLIN_DIR'])

def load_merlin():
    ckpt = hf_hub_download(MERLIN_REPO, MERLIN_CKPT, revision=MERLIN_REVISION, local_dir=MERLIN_DIR)
    state = torch.load(ckpt, map_location='cpu')
    prefix = 'encode_image.i3_resnet.'
    enc = i3res.I3ResNet(copy.deepcopy(torchvision.models.resnet152(weights=None)), class_nb=1692, conv_class=True,
                         ImageEmbedding=True)
    enc.load_state_dict({k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}, strict=True)
    return enc.eval()

# CT-FM preprocessing (authors' script), split so that the background-only version can use the original's crop box
CTFM_LOAD = Compose([LoadImage(ensure_channel_first=True), EnsureType(), Orientation(axcodes='SPL'),
                     Spacing(pixdim=CTFM_SPACING, mode='bilinear')])
CTFM_SCALE = ScaleIntensityRange(a_min=-1024, a_max=2048, b_min=0, b_max=1, clip=True)

def ctfm_inputs(paths, crop_from=None):
    """Preprocessed CT-FM inputs for each variant. With crop_from, every variant uses that variant's foreground box."""
    imgs = {v: CTFM_LOAD(str(p)) for v, p in paths.items()}
    out, boxes = {}, {}
    for v, img in imgs.items():
        src = imgs[crop_from] if crop_from else img
        start, end = generate_spatial_bounding_box(src, select_fn=lambda x: x > 0)   # CropForeground's default
        out[v] = _plain(CTFM_SCALE(SpatialCrop(roi_start=start, roi_end=end)(img)))
        boxes[v] = [int(a) for a in start] + [int(b) for b in end]
    return out, boxes

def _plain(t):
    return (t.as_tensor() if hasattr(t, 'as_tensor') else torch.as_tensor(t)).float()

def merlin_inputs(paths):
    return {v: _plain(MERLIN_TRANSFORMS({'image': str(p)})['image']) for v, p in paths.items()}

class PrepDataset(torch.utils.data.Dataset):
    """CPU preprocessing in DataLoader workers; the GPU only runs the encoders."""
    def __init__(self, items):
        self.items = items              # (key, {variant: path}, crop_from)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        key, paths, crop_from = self.items[i]
        try:
            ct, boxes = ctfm_inputs(paths, crop_from)
            return {'key': key, 'ctfm': ct, 'merlin': merlin_inputs(paths), 'boxes': boxes, 'error': None}
        except Exception as e:
            root = e
            while root.__cause__ is not None:      # MONAI wraps the real error; report the root cause
                root = root.__cause__
            return {'key': key, 'error': f'{e!r} | root cause: {root!r}'[:600]}

@torch.no_grad()
def ctfm_embed(model, vol):
    feats, batch = [], []
    def run(b):
        x = torch.cat(b).to(DEVICE)
        return torch.nn.functional.adaptive_avg_pool3d(model(x)[-1], 1).flatten(1).cpu()
    for patch, _ in SlidingWindowSplitter(CTFM_PATCH, CTFM_OVERLAP)(vol.unsqueeze(0)):
        batch.append(patch)
        if len(batch) == CTFM_BATCH:
            feats.append(run(batch)); batch = []
    if batch:
        feats.append(run(batch))
    f = torch.cat(feats)
    return f.mean(0).numpy().astype(np.float32), int(len(f))

@torch.no_grad()
def merlin_embed(model, vol):
    return model(vol.unsqueeze(0).to(DEVICE)).reshape(-1).cpu().numpy().astype(np.float32)

def embed_items(items, out_path_of, models, extra_of=None, label='scans'):
    """Embed every item with both encoders and save one .npz per item (resume point)."""
    loader = torch.utils.data.DataLoader(PrepDataset(items), batch_size=None, shuffle=False,
                                         num_workers=EMB_WORKERS if len(items) > 1 else 0)
    done, failed, t0 = 0, [], time.time()
    for rec in loader:
        if rec['error']:
            failed.append({'key': rec['key'], 'error': rec['error']})
            continue
        arrays, meta = {}, {'emb_key': EMB_KEY, 'boxes': rec['boxes'], **(extra_of(rec['key']) if extra_of else {})}
        for v, vol in rec['ctfm'].items():
            arrays[f'ctfm__{v}'], meta[f'ctfm__{v}__patches'] = ctfm_embed(models['ctfm'], vol)
        for v, vol in rec['merlin'].items():
            arrays[f'merlin__{v}'] = merlin_embed(models['merlin'], vol)
        path = out_path_of(rec['key'])
        tmp = path.with_name(path.stem + '.tmp.npz')
        np.savez(tmp, meta=json.dumps(meta, default=_jsonable), **arrays)
        tmp.replace(path)
        done += 1
        if done % 25 == 0 or done == len(items):
            print(f'  {label}: {done}/{len(items)} embedded ({(time.time() - t0) / 60:.1f} min)')
    return failed

def emb_path(key):
    return CACHE / 'embeddings' / f'{key}.npz'

def emb_ok(path, variants):
    if not path.exists():
        return False
    with np.load(path) as z:
        meta = json.loads(str(z['meta']))
        return meta.get('emb_key') == EMB_KEY and all(f'{e}__{v}' in z.files for e in ENCODERS for v in variants)

In [ ]:
# 9. Part 1: prepare and embed every scan (label-free; resumable)
def fake_part1():
    """Local code tests only: random embeddings and headers instead of images (seeded, label-free)."""
    rng = np.random.default_rng(7)
    for key in PATIENTS.key:
        if not header_ok(key):
            write_json_atomic(header_path(key), {
                'key': key, 'prep_key': PREP_KEY, 'hdr_z_spacing': float(rng.choice([0.625, 1.25, 2.5, 5.0])),
                'hdr_slice_thickness': float(rng.choice([1.25, 2.5, 5.0])), 'hdr_pixel_spacing': float(rng.uniform(.6, .9)),
                'hdr_recon_diameter': float(rng.uniform(300, 450)), 'hdr_kvp': float(rng.choice([100, 120, 140])),
                'hdr_tube_current': float(rng.uniform(100, 400)), 'hdr_exposure': np.nan, 'hdr_ctdivol': np.nan,
                'hdr_pitch': float(rng.choice([0.984, 1.375])), 'hdr_kernel': str(rng.choice(['B30F', 'STANDARD', 'B'])),
                'hdr_filter_type': np.nan, 'hdr_patient_position': 'FFS', 'hdr_contrast_agent_given': 'no',
                'hdr_nonuniform_spacing': False, 'body_area_cm2': float(rng.uniform(400, 900))})
        if not emb_ok(emb_path(key), LABEL_FREE_VARIANTS):
            arrays = {f'ctfm__{v}': rng.normal(size=512).astype(np.float32) for v in LABEL_FREE_VARIANTS}
            arrays.update({f'merlin__{v}': np.abs(rng.normal(size=2048)).astype(np.float32) for v in LABEL_FREE_VARIANTS})
            np.savez(emb_path(key), meta=json.dumps({'emb_key': EMB_KEY, 'fake': True}), **arrays)

t_part1 = time.time()
prep_failed, emb_failed = [], []
if TEST.get('fake_images'):
    fake_part1()
    MODELS = None
else:
    # 9a. download and prepare the scans that are not finished yet
    todo = [r for r in PATIENTS.itertuples() if not emb_ok(emb_path(r.key), LABEL_FREE_VARIANTS)
            or not header_ok(r.key)]
    print(f'Scans to prepare: {len(todo)} of {len(PATIENTS)} (the others are finished in the cache)')
    if todo:
        res = Parallel(n_jobs=PREP_JOBS, return_as='generator_unordered')(
            delayed(prepare_patient)(r.key, r.SeriesInstanceUID, r.FileSize) for r in todo)
        for i, r in enumerate(res, 1):
            if r['status'] == 'failed':
                prep_failed.append(r)
                print(f"  FAILED {r['key']}: {r['error']}")
            if i % 25 == 0 or i == len(todo):
                print(f'  prepared {i}/{len(todo)} ({(time.time() - t_part1) / 60:.1f} min, '
                      f'{shutil.disk_usage(WORK).free / 1e9:.0f} GB free)')
    # 9b. embed the prepared scans that have no embeddings yet
    MODELS = {'ctfm': load_ctfm().to(DEVICE), 'merlin': load_merlin().to(DEVICE)}
    todo = [k for k in PATIENTS.key if not emb_ok(emb_path(k), LABEL_FREE_VARIANTS)
            and all(p.exists() for p in nifti_paths(k).values())]
    print(f'Scans to embed: {len(todo)}')
    items = [(k, {v: nifti_paths(k)[v] for v in LABEL_FREE_VARIANTS}, 'original') for k in todo]
    emb_failed = embed_items(items, emb_path, MODELS, label='original + background')
    for f in emb_failed:
        print(f"  EMBEDDING FAILED {f['key']}: {f['error']}")
print(f'Part 1 finished in {(time.time() - t_part1) / 60:.1f} min')

In [ ]:
# 10. Check against the official Merlin class (once), and the image inventory
merlin_check_file = CACHE / 'merlin_official_check.json'
if MODELS is not None and not merlin_check_file.exists() and not TEST.get('skip_official_check'):
    try:
        k0 = next(k for k in PATIENTS.key if nifti_paths(k)['original'].exists())
        x = merlin_inputs({'original': nifti_paths(k0)['original']})['original']
        official = merlin.Merlin(ImageEmbedding=True).eval().to(DEVICE)
        with torch.no_grad():
            a = official(x.unsqueeze(0).to(DEVICE)).reshape(-1).cpu().numpy()
        b = merlin_embed(MODELS['merlin'], x)
        check = {'patient': k0, 'max_abs_diff': float(np.abs(a - b).max()), 'max_abs_value': float(np.abs(a).max())}
        del official
    except Exception as e:
        check = {'error': repr(e)[:400]}
    write_json_atomic(merlin_check_file, check)
if merlin_check_file.exists():
    print('Merlin official-class check:', json.loads(merlin_check_file.read_text()))

hdr = pd.DataFrame([json.loads(header_path(k).read_text()) for k in PATIENTS.key if header_path(k).exists()],
                   columns=None)
if 'key' not in hdr:
    hdr = pd.DataFrame(columns=['key'])
complete = {k for k in PATIENTS.key if emb_ok(emb_path(k), LABEL_FREE_VARIANTS)}
inventory = PATIENTS[['key', 'collection', 'PatientID', 'SeriesInstanceUID', 'ImageCount', 'FileSize']].merge(
    hdr.drop(columns=['SeriesInstanceUID'], errors='ignore'), on='key', how='left')
inventory['embedded'] = inventory.key.isin(complete)
inventory.to_csv(CACHE / 'image_inventory.csv', index=False)
pd.DataFrame(prep_failed + emb_failed, columns=['key', 'status', 'error']).to_csv(CACHE / 'failures_this_session.csv',
                                                                                  index=False)
cov = pts.assign(embedded=pts.key.isin(complete)).groupby('task_id').embedded.agg(['sum', 'size', 'mean'])
show(cov.rename(columns={'sum': 'embedded', 'size': 'patients', 'mean': 'coverage'}))
if not TEST.get('fake_images') and len(hdr):
    print('Slices in the download vs TCIA image count, median ratio:',
          round(float((inventory.hdr_n_slices / inventory.ImageCount).median()), 3))
    print('Non-uniform slice spacing:', int(inventory.hdr_nonuniform_spacing.eq(True).sum()),
          '| gantry tilt > 1 degree:', int((inventory.hdr_gantry_tilt_deg > 1).sum()))
if (cov['mean'] < GATE_MIN_COVERAGE).any():
    print(f'WARNING: fewer than {GATE_MIN_COVERAGE:.0%} of the patients of a task have embeddings. '
          'Run all again (failed downloads are retried); if it persists, inspect the failed-download log.')
if TEST.get('stop_after_images'):
    raise SystemExit('Local test: stopping after the image part.')

## Part 2 and 3 · The analysis (once on shuffled labels as a rehearsal, then on the real labels)

In [ ]:
# 11. Model building blocks (identical to notebooks 03 and 04), plus the same models on embeddings
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    return [list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y))
            for r in range(N_REPEATS)]

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, PA, PB):
    """Repeat-averaged AUC(A) - AUC(B) on the same patients, with a paired bootstrap 95% CI."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(np.mean([auc(y[idx], PA[r][idx]) - auc(y[idx], PB[r][idx]) for r in range(PA.shape[0])]))
    return mean_auc(y, PA) - mean_auc(y, PB), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_mean(x):
    rng = np.random.default_rng(SEED)
    x = np.asarray(x, float)
    vals = [x[rng.choice(len(x), len(x), replace=True)].mean() for _ in range(N_BOOT)]
    return float(x.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# --- metadata models (notebook 03) ---
def _fit_fold_df(df, y, cols, tr, te, k):
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(df.iloc[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[te])
    return te, proba

def cv_oof_df(df, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_df)(df, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- the same logistic regression on embeddings (standardized inside each training fold) ---
def _fit_fold_matrix(X, y, tr, te, k, extras):
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(X[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(X[te])
    out = {}
    for name, Xe in extras.items():           # the fitted fold model applied to other inputs of the same patients
        pe = np.full((len(te), k), np.nan)
        ok = np.where(~np.isnan(Xe[te]).any(axis=1))[0]
        if len(ok):
            tmp = np.zeros((len(ok), k))
            tmp[:, m.classes_] = m.predict_proba(Xe[te][ok])
            pe[ok] = tmp
        out[name] = pe
    return te, proba, out

def cv_oof_matrix(X, y, splits, extras=None):
    extras = extras or {}
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_matrix)(X, y, tr, te, k, extras) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    E = {n: np.full((len(splits), len(y), k), np.nan) for n in extras}
    for (r, _, _), (te, proba, out) in zip(jobs, res):
        P[r, te] = proba
        for n, pe in out.items():
            E[n][r, te] = pe
    return P, E

def _fit_fold_combined(df, E, y, cols, tr, te, k):
    ct = column_transformer(cols).fit(df.iloc[tr])
    sc = StandardScaler().fit(E[tr])
    Xtr = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[tr])), sparse.csr_matrix(sc.transform(E[tr]))]).tocsr()
    Xte = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[te])), sparse.csr_matrix(sc.transform(E[te]))]).tocsr()
    m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(Xte)
    return te, proba

def cv_oof_combined(df, E, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_combined)(df, E, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- exact permutation tests on one fixed label-independent split (notebook 03, 7b.6) ---
def encoded_folds_df(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def scaled_folds(X):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(X):
        sc = StandardScaler().fit(X[tr])
        out.append((tr, te, sc.transform(X[tr]), sc.transform(X[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(folds, y):
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

# --- within-group AUC and its exact test (notebook 04, part D) ---
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(folds, y, strata, keep):
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

In [ ]:
# 12. Reproduction check: notebook 03's metadata model on these four tasks (real labels; only known numbers)
def task_frame(tid, shuffle_seed=None):
    """Task patients with metadata, groups, headers and body size. shuffle_seed: shuffled labels (rehearsal)."""
    df = labels[labels.task_id == tid].merge(feat, on=KEY, how='left', validate='one_to_one')
    df['key'] = df.collection + '__' + df.PatientID
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if shuffle_seed is not None:
        y = np.random.default_rng(shuffle_seed).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(polish, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

repro = []
for tid in TASKS:
    df, y, classes = task_frame(tid)
    P = cv_oof_df(df, y, SCANNER_COLS, repeated_splits(y))
    o = oof03[oof03.task_id == tid]
    diff = max(np.abs(o[o.repeat == r].set_index('PatientID').reindex(df.PatientID)[['p_0', 'p_1']].to_numpy()
                      - P[r]).max() for r in range(N_REPEATS))
    repro.append({'task_id': tid, 'nb03_auc': a03.set_index('task_id').auc[tid], 'nb05_auc': mean_auc(y, P),
                  'oof_max_abs_diff': float(diff), 'reproduced': bool(diff < 1e-6)})
repro = pd.DataFrame(repro)
show(repro)
REPRODUCED = bool(repro.reproduced.all())
if not REPRODUCED:
    print('WARNING: notebook 03 was not reproduced exactly (package versions may have changed). '
          'Review the run log before interpreting any result below.')

In [ ]:
# 13. Part 2: the acquisition swap (label-dependent: which class is thinner), embedded with both encoders
def swap_volumes(key, target_z, out_dir):
    """Thin -> thick: slab-average along z to the target spacing and resample; plus the matched in-plane control."""
    paths = nifti_paths(key)
    img = sitk.ReadImage(str(paths['original']), sitk.sitkFloat32)
    arr = sitk.GetArrayFromImage(img)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(paths['mask']))).astype(bool)
    sx, sy, own = img.GetSpacing()
    w = max(1, int(round(target_z / own)))
    slab = like(ndimage.uniform_filter1d(arr, size=w, axis=0, mode='nearest'), img, np.float32)
    nz = int(np.floor((arr.shape[0] - 1) * own / target_z)) + 1
    swap = sitk.Resample(slab, [arr.shape[2], arr.shape[1], nz], sitk.Transform(), sitk.sitkLinear, img.GetOrigin(),
                         (sx, sy, target_z), img.GetDirection(), -1024.0, sitk.sitkFloat32)
    back = sitk.GetArrayFromImage(sitk.Resample(swap, img, sitk.Transform(), sitk.sitkLinear, -1024.0))
    d_swap = float(np.abs(back - arr)[mask].mean())
    best = None
    for s in CONTROL_SIGMAS_MM:
        g = ndimage.gaussian_filter(arr, sigma=(0, s / sy, s / sx))
        d = float(np.abs(g - arr)[mask].mean())
        if best is None or abs(d - d_swap) < best[0]:
            best = (abs(d - d_swap), s, d, g)
    out = {'swap': out_dir / f'{key}__swap.nii.gz', 'control': out_dir / f'{key}__control.nii.gz'}
    sitk.WriteImage(like(sitk.GetArrayFromImage(swap), swap), str(out['swap']), useCompression=True)
    sitk.WriteImage(like(best[3], img), str(out['control']), useCompression=True)
    return out, {'own_z': own, 'target_z': target_z, 'slab_slices': w, 'mean_abs_change_swap': d_swap,
                 'control_sigma_mm': best[1], 'mean_abs_change_control': best[2]}

def safe_swap_volumes(key, target_z, out_dir):
    try:
        return swap_volumes(key, target_z, out_dir)
    except Exception as e:      # one unreadable scan must not stop the run; it is listed and left out
        return repr(e)[:400]

def swap_plan(tid, y, classes, h):
    """Which class is thinner, its target, and the patients to swap (None if not applicable)."""
    z = h.hdr_z_spacing.to_numpy()
    med = [float(np.nanmedian(z[y == j])) for j in range(len(classes))]
    if TEST.get('swap_target_mm'):
        thin, target = 0, TEST['swap_target_mm']
    elif abs(med[0] - med[1]) < SWAP_MIN_DIFF_MM:
        return {'applicable': False, 'medians': med}
    else:
        thin = int(np.argmin(med)); target = med[1 - thin]
    sel = np.where((y == thin) & (z < target - SWAP_MIN_GAP_MM))[0]
    return {'applicable': True, 'medians': med, 'thin_class': thin, 'thick_class': 1 - thin, 'target_z': target,
            'rows': sel}

def run_swaps(mode, plans, frames):
    """Prepare and embed the swap and control scans for every applicable task (resumable, per mode)."""
    sdir = OUT_ROOT / mode / 'swap_embeddings'
    sdir.mkdir(parents=True, exist_ok=True)
    vdir = WORK / f'swap_{mode}'
    vdir.mkdir(exist_ok=True)
    jobs = []
    for tid, plan in plans.items():
        if not plan['applicable']:
            continue
        df = frames[tid]
        for i in plan['rows']:
            key = df.key[i]
            p = sdir / f"{key}__{plan['target_z']:.3f}.npz"
            if not emb_ok(p, ('swap', 'control')):
                jobs.append((key, plan['target_z'], p))
    if not jobs:
        return sdir
    if TEST.get('fake_images'):
        rng = np.random.default_rng(11)
        for key, t, p in jobs:
            with np.load(emb_path(key)) as z:
                arrays = {f'{e}__{v}': z[f'{e}__original'] + rng.normal(scale=0.1, size=z[f'{e}__original'].shape
                                                                        ).astype(np.float32)
                          for e in ENCODERS for v in ('swap', 'control')}
            np.savez(p, meta=json.dumps({'emb_key': EMB_KEY, 'fake': True, 'mean_abs_change_swap': 1.0,
                                         'mean_abs_change_control': 1.0, 'control_sigma_mm': 0.5}), **arrays)
        return sdir
    # original volumes are needed again: re-download any that are gone (e.g. after a disconnect)
    missing = [k for k, _, _ in jobs if not all(q.exists() for q in nifti_paths(k).values())]
    if missing:
        print(f'  re-preparing {len(missing)} scans for the swap')
        r = PATIENTS.set_index('key').loc[missing]
        Parallel(n_jobs=PREP_JOBS)(delayed(prepare_patient)(k, u, s) for k, u, s in
                                   zip(missing, r.SeriesInstanceUID, r.FileSize))
    print(f'  making {len(jobs)} swap + control scans')
    made = Parallel(n_jobs=PREP_JOBS)(delayed(safe_swap_volumes)(k, t, vdir) for k, t, _ in jobs)
    for (k, _, _), m in zip(jobs, made):
        if isinstance(m, str):
            print(f'  SWAP FAILED {k}: {m}')
    ok = [(j, m) for j, m in zip(jobs, made) if not isinstance(m, str)]
    info = {k: m[1] for (k, _, _), m in ok}
    path_of = {k: p for (k, _, p), _ in ok}
    items = [(k, m[0], None) for (k, _, _), m in ok]
    failed = embed_items(items, lambda k: path_of[k], MODELS, extra_of=lambda k: info[k], label='swap + control')
    for f in failed:
        print(f"  SWAP EMBEDDING FAILED {f['key']}: {f['error']}")
    shutil.rmtree(vdir, ignore_errors=True)
    return sdir

In [ ]:
# 14. Part 3: every analysis for one task, one encoder
def load_embeddings(keys, variants, path_of):
    out = {(e, v): [] for e in ENCODERS for v in variants}
    for k in keys:
        p = path_of(k)
        ok = p is not None and p.exists()
        z = np.load(p) if ok else None
        for e in ENCODERS:
            for v in variants:
                name = f'{e}__{v}'
                out[(e, v)].append(z[name] if ok and name in z.files else None)
        if z is not None:
            z.close()
    res = {}
    for (e, v), lst in out.items():
        d = next((len(a) for a in lst if a is not None), 1)
        res[(e, v)] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in lst]).astype(float)
    return res

def flag(lo, q):
    return bool(lo >= FLAG_LOWER_CI and q < FLAG_Q)

def analyse_task(tid, mode, shuffle_seed, swap_dir):
    df, y, classes = task_frame(tid, shuffle_seed)
    hdr_t = pd.DataFrame([json.loads(header_path(k).read_text()) if header_path(k).exists() else {'key': k}
                          for k in df.key])
    wanted = ['body_area_cm2', 'hdr_contrast_agent_given', 'hdr_nonuniform_spacing', 'hdr_coverage_mm',
              'hdr_n_slices'] + HDR_COLS
    df = df.merge(hdr_t[['key'] + [c for c in wanted if c in hdr_t]], on='key', how='left')
    for c in wanted:
        if c not in df:
            df[c] = np.nan
    for c in HDR_CATEGORICAL:
        df[c] = df[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
    has = df.key.isin(complete).to_numpy()
    n_all = len(df)
    df, y = df[has].reset_index(drop=True), y[has]
    splits = repeated_splits(y)
    base = {'task_id': tid, 'n_patients': len(y), 'n_excluded_no_image': int(n_all - len(y)),
            'classes': ' | '.join(f'{c}: {int((y == j).sum())}' for j, c in enumerate(classes))}
    emb = load_embeddings(df.key, LABEL_FREE_VARIANTS, emb_path)
    grouping = TASKS[tid]
    strata = {'hospital': df.site.astype(object).where(df.site.notna(), '').astype(str).to_numpy(),
              'description language': df.language.to_numpy(),
              'scanner vendor': df.vendor.astype(object).where(df.vendor.notna(), '').astype(str).to_numpy()}[grouping]
    keep = strata != ''
    plan = swap_plan(tid, y, classes, df) if swap_dir is not None else {'applicable': False, 'medians': [np.nan] * 2}
    swap_emb = None
    if plan['applicable']:
        paths = {df.key[i]: swap_dir / f"{df.key[i]}__{plan['target_z']:.3f}.npz" for i in plan['rows']}
        swap_emb = load_embeddings(df.key, ('swap', 'control'), lambda k: paths.get(k))

    # metadata model on the same patients (for the comparison) and body size alone
    P_meta = cv_oof_df(df, y, SCANNER_COLS, splits)
    P_body = cv_oof_df(df, y, ['body_area_cm2'], splits)
    lo_b, hi_b = boot_ci(y, P_body)
    rec = {'probe': [], 'within': [], 'heldout': [], 'meta': [], 'swap': [], 'oof': []}
    for enc in ENCODERS:
        X, Xb = emb[(enc, 'original')], emb[(enc, 'background')]
        extras = {'background_frozen': Xb}
        if swap_emb is not None:
            extras.update({'swap': swap_emb[(enc, 'swap')], 'control': swap_emb[(enc, 'control')]})
        # (3) the image probe, and (5a) the background-only probe
        P, E = cv_oof_matrix(X, y, splits, extras)
        lo, hi = boot_ci(y, P)
        obs, p, null = permutation_test(scaled_folds(X), y)
        Pb, _ = cv_oof_matrix(Xb, y, splits)
        lo2, hi2 = boot_ci(y, Pb)
        obs2, p2, null2 = permutation_test(scaled_folds(Xb), y)
        rec['probe'].append({**base, 'encoder': enc, 'embedding_dim': X.shape[1],
                             'image_auc': mean_auc(y, P), 'image_ci_low': lo, 'image_ci_high': hi,
                             'image_perm_p': p, 'image_null_mean_auc': float(null.mean()),
                             'background_auc': mean_auc(y, Pb), 'background_ci_low': lo2, 'background_ci_high': hi2,
                             'background_perm_p': p2, 'background_null_mean_auc': float(null2.mean()),
                             'image_probe_on_background_auc': mean_auc(y, E['background_frozen']),
                             'body_size_auc': mean_auc(y, P_body), 'body_size_ci_low': lo_b, 'body_size_ci_high': hi_b,
                             'metadata_auc_same_patients': mean_auc(y, P_meta)})
        # (4a) within-group AUC and its exact test
        w = within_site_estimate(y[keep], P[:, keep], strata[keep])
        if w['n_pairs']:
            obs_w, p_w, null_w = within_group_permutation_test(scaled_folds(X), y, strata, keep)
        else:
            obs_w, p_w, null_w = np.nan, np.nan, np.array([np.nan])
        rec['within'].append({'task_id': tid, 'encoder': enc, 'grouping': grouping, 'n_patients': int(keep.sum()),
                              'n_groups': len(np.unique(strata[keep])),
                              'n_groups_with_both_classes': w['n_sites_with_pairs'], 'n_pairs': w['n_pairs'],
                              'within_auc': w['auc'], 'within_ci_low': w['ci_low'], 'within_ci_high': w['ci_high'],
                              'overall_auc_same_patients': mean_auc(y[keep], P[:, keep]),
                              'within_auc_fixed_split': obs_w, 'null_mean_within_auc': float(np.nanmean(null_w)),
                              'null_95th_within_auc': float(np.nanpercentile(null_w, 95)) if np.isfinite(null_w).any()
                              else np.nan, 'perm_p': p_w})
        # (4b) whole hospitals held out
        if grouping == 'hospital':
            hrow = {'task_id': tid, 'encoder': enc, 'n_patients': int(keep.sum()),
                    'n_hospitals': len(np.unique(strata[keep]))}
            Xk, yk, sk = X[keep], y[keep], strata[keep]
            gkf = list(GroupKFold(min(N_SPLITS, len(np.unique(sk)))).split(np.zeros(len(yk)), yk, sk))
            if all(len(np.unique(yk[tr])) == 2 for tr, _ in gkf):
                Pg, _ = cv_oof_matrix(Xk, yk, [gkf])
                lo_g, hi_g = boot_ci(yk, Pg)
                hrow.update({'heldout_auc': mean_auc(yk, Pg), 'heldout_ci_low': lo_g, 'heldout_ci_high': hi_g})
            else:
                hrow['heldout_note'] = 'a training fold had one class only'
            rec['heldout'].append(hrow)
        # (4c) beyond metadata
        P_comb = cv_oof_combined(df, X, y, SCANNER_COLS, splits)
        d, d_lo, d_hi = boot_diff(y, P_comb, P_meta)
        mrow = {'task_id': tid, 'encoder': enc, 'metadata_auc': mean_auc(y, P_meta), 'image_auc': mean_auc(y, P),
                'metadata_plus_image_auc': mean_auc(y, P_comb), 'added_by_image': d, 'added_ci_low': d_lo,
                'added_ci_high': d_hi}
        si, sm = P[:, :, 1].mean(0), P_meta[:, :, 1].mean(0)
        for j, c in enumerate(classes):
            m = np.where(y == j)[0]
            rng = np.random.default_rng(SEED)
            boots = []
            for _ in range(N_BOOT):
                b = rng.choice(m, len(m), replace=True)
                boots.append(spearmanr(si[b], sm[b]).statistic)
            mrow.update({f'spearman_class{j}': float(spearmanr(si[m], sm[m]).statistic), f'class{j}': c,
                         f'spearman_class{j}_ci_low': float(np.nanpercentile(boots, 2.5)),
                         f'spearman_class{j}_ci_high': float(np.nanpercentile(boots, 97.5))})
        rec['meta'].append(mrow)
        # (5b) the acquisition swap, with the frozen fold models
        srow = {'task_id': tid, 'encoder': enc, 'applicable': plan['applicable'],
                f'median_z_spacing_{classes[0]}': plan['medians'][0],
                f'median_z_spacing_{classes[1]}': plan['medians'][1]}
        rows_ = plan['rows'][~np.isnan(E['swap'][0, plan['rows'], 0])] if plan['applicable'] else []
        if plan['applicable'] and len(rows_) < 2:
            srow['note'] = 'no swapped scan could be embedded'
        if plan['applicable'] and len(rows_) >= 2:
            tc = plan['thick_class']
            sh = (E['swap'][:, rows_, tc] - P[:, rows_, tc]).mean(0)
            ch = (E['control'][:, rows_, tc] - P[:, rows_, tc]).mean(0)
            m_s, lo_s, hi_s = boot_mean(sh)
            m_c, lo_c, hi_c = boot_mean(ch)
            m_d, lo_d, hi_d = boot_mean(sh - ch)
            metas = []
            for k in df.key[rows_]:
                with np.load(swap_dir / f"{k}__{plan['target_z']:.3f}.npz") as z:
                    metas.append(json.loads(str(z['meta'])))
            srow.update({'thin_class': classes[plan['thin_class']], 'thick_class': classes[tc],
                         'target_z_spacing': plan['target_z'], 'n_swapped': len(rows_),
                         'shift_toward_thick': m_s, 'shift_ci_low': lo_s, 'shift_ci_high': hi_s,
                         'control_shift': m_c, 'control_ci_low': lo_c, 'control_ci_high': hi_c,
                         'swap_minus_control': m_d, 'difference_ci_low': lo_d, 'difference_ci_high': hi_d,
                         'median_abs_hu_change_swap': float(np.median([q['mean_abs_change_swap'] for q in metas])),
                         'median_abs_hu_change_control': float(np.median([q['mean_abs_change_control'] for q in metas])),
                         'rule_met': bool(m_s >= SWAP_SHIFT_MIN and lo_s > 0 and lo_d > 0)})
        else:
            srow['rule_met'] = False
        rec['swap'].append(srow)
        rec['oof'].append(pd.DataFrame({'task_id': tid, 'encoder': enc, 'key': df.key, 'label': y,
                                        'p_image': P[:, :, 1].mean(0), 'p_background': Pb[:, :, 1].mean(0),
                                        'p_metadata': P_meta[:, :, 1].mean(0)}))

    # (6) DICOM-header audit (secondary, estimates only)
    P_hdr = cv_oof_df(df, y, SCANNER_COLS + HDR_COLS, splits)
    P_hdr_only = cv_oof_df(df, y, HDR_COLS, splits)
    d, d_lo, d_hi = boot_diff(y, P_hdr, P_meta)
    lo_h, hi_h = boot_ci(y, P_hdr)
    lo_ho, hi_ho = boot_ci(y, P_hdr_only)
    lo_m, hi_m = boot_ci(y, P_meta)
    hdr_row = {**base, 'api_auc': mean_auc(y, P_meta), 'api_ci_low': lo_m, 'api_ci_high': hi_m,
               'api_plus_header_auc': mean_auc(y, P_hdr), 'api_plus_header_ci_low': lo_h,
               'api_plus_header_ci_high': hi_h, 'header_only_auc': mean_auc(y, P_hdr_only),
               'header_only_ci_low': lo_ho, 'header_only_ci_high': hi_ho, 'added_by_header': d,
               'added_ci_low': d_lo, 'added_ci_high': d_hi,
               **{f'filled_{c}': float(df[c].notna().mean()) for c in HDR_COLS},
               # described by class, never used as inputs (clinical choice or scan irregularity)
               'contrast_agent_given_by_class': ' | '.join(
                   f"{c}: {float((df.hdr_contrast_agent_given[y == j] == 'yes').mean()):.2f}"
                   for j, c in enumerate(classes)),
               'nonuniform_spacing_by_class': ' | '.join(
                   f"{c}: {int(df.hdr_nonuniform_spacing[y == j].eq(True).sum())}" for j, c in enumerate(classes)),
               'median_coverage_mm_by_class': ' | '.join(
                   f"{c}: {df.hdr_coverage_mm[y == j].median():.0f}" for j, c in enumerate(classes)),
               'median_z_spacing_by_class': ' | '.join(
                   f"{c}: {df.hdr_z_spacing[y == j].median():.2f}" for j, c in enumerate(classes))}
    return rec, hdr_row

In [ ]:
# 15. Run one mode (rehearsal on shuffled labels, or real), with BH, flags and saving
def run_mode(mode):
    out = OUT_ROOT / mode
    (out / 'per_task').mkdir(parents=True, exist_ok=True)
    shuffle_seed = 999 if mode == 'rehearsal' else (1000 if TEST.get('real_is_shuffled') else None)
    key = hashlib.md5(json.dumps({'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT,
                                  'N_PERM': N_PERM, 'LR': LR_PARAMS, 'EMB_KEY': EMB_KEY, 'TEST': TEST,
                                  'shuffle': shuffle_seed, 'swap': [SWAP_MIN_DIFF_MM, SWAP_MIN_GAP_MM,
                                                                    CONTROL_SIGMAS_MM],
                                  'complete': sorted(complete)}, sort_keys=True, default=str).encode()).hexdigest()
    print(f'\n===== {mode.upper()} pass ({"shuffled labels" if shuffle_seed is not None else "REAL labels"}) =====')
    frames, plans = {}, {}
    for tid in TASKS:
        df, y, classes = task_frame(tid, shuffle_seed)
        h = pd.DataFrame([json.loads(header_path(k).read_text()) if header_path(k).exists() else {'key': k}
                          for k in df.key])
        has = df.key.isin(complete).to_numpy()
        df = df[has].reset_index(drop=True).merge(h[['key', 'hdr_z_spacing']], on='key', how='left')
        frames[tid] = df
        plans[tid] = swap_plan(tid, y[has], classes, df)
        print(f"  swap {tid}: medians {np.round(plans[tid]['medians'], 2)} -> "
              f"{'applicable, ' + str(len(plans[tid]['rows'])) + ' scans' if plans[tid]['applicable'] else 'not applicable'}")
    swap_dir = run_swaps(mode, plans, frames)
    allrec = {'probe': [], 'within': [], 'heldout': [], 'meta': [], 'swap': []}
    hdr_rows, oof = [], []
    t_all = time.time()
    for i, tid in enumerate(TASKS, 1):
        t0 = time.time()
        stem = re.sub(r'\W+', '_', tid).strip('_')
        saved = out / 'per_task' / f'{stem}.json'
        if saved.exists() and json.loads(saved.read_text()).get('settings_key') == key:
            r = json.loads(saved.read_text())
            rec, hrow = r['rec'], r['hdr']
            oof.append(pd.read_csv(out / 'per_task' / f'{stem}_oof.csv.gz'))
            print(f'{i}/{len(TASKS)} {tid:<36} loaded from an earlier run')
        else:
            rec, hrow = analyse_task(tid, mode, shuffle_seed, swap_dir)
            o = pd.concat(rec.pop('oof'), ignore_index=True)
            o.to_csv(out / 'per_task' / f'{stem}_oof.csv.gz', index=False)
            oof.append(o)
            write_json_atomic(saved, {'settings_key': key, 'rec': rec, 'hdr': hrow})
            pr = {r_['encoder']: r_ for r_ in rec['probe']}
            print(f"{i}/{len(TASKS)} {tid:<36} CT-FM {pr['ctfm']['image_auc']:.3f} (bg {pr['ctfm']['background_auc']:.3f})"
                  f" | Merlin {pr['merlin']['image_auc']:.3f} (bg {pr['merlin']['background_auc']:.3f})"
                  f"  ({(time.time() - t0) / 60:.1f} min)")
        for k_ in allrec:
            allrec[k_] += rec[k_]
        hdr_rows.append(hrow)
    print(f'Analysis finished in {(time.time() - t_all) / 60:.1f} min')

    probe = pd.DataFrame(allrec['probe'])
    for col in ('image', 'background'):
        probe[f'{col}_perm_q_bh'] = np.nan
        for enc, idx in probe.groupby('encoder').groups.items():
            probe.loc[idx, f'{col}_perm_q_bh'] = bh(probe.loc[idx, f'{col}_perm_p'])
        probe[f'{col}_flagged'] = [flag(lo, q) for lo, q in zip(probe[f'{col}_ci_low'], probe[f'{col}_perm_q_bh'])]
    within = pd.DataFrame(allrec['within'])
    within['perm_q_bh'] = np.nan
    for enc, idx in within.groupby('encoder').groups.items():
        within.loc[idx, 'perm_q_bh'] = bh(within.loc[idx, 'perm_p'])
    swap = pd.DataFrame(allrec['swap'])
    wording = probe[['task_id', 'encoder', 'image_flagged', 'background_flagged']].merge(
        swap[['task_id', 'encoder', 'rule_met']].rename(columns={'rule_met': 'swap_rule_met'}), on=['task_id', 'encoder'])
    wording['shortcut_wording_allowed'] = wording.image_flagged & (wording.background_flagged | wording.swap_rule_met)
    tables = {'image_probe.csv': probe, 'image_within_group.csv': within,
              'image_heldout_hospitals.csv': pd.DataFrame(allrec['heldout']),
              'image_vs_metadata.csv': pd.DataFrame(allrec['meta']), 'image_swap.csv': swap,
              'header_audit.csv': pd.DataFrame(hdr_rows), 'shortcut_wording.csv': wording}
    for name, t in tables.items():
        t.to_csv(out / name, index=False)
    pd.concat(oof, ignore_index=True).to_csv(out / 'oof_image_predictions.csv.gz', index=False)
    repro.to_csv(out / 'reproduction_check.csv', index=False)
    show(probe[['task_id', 'encoder', 'n_patients', 'image_auc', 'image_ci_low', 'image_perm_q_bh', 'image_flagged',
                'background_auc', 'background_ci_low', 'background_perm_q_bh', 'background_flagged', 'body_size_auc']], 20)
    show(within[['task_id', 'encoder', 'grouping', 'n_pairs', 'within_auc', 'null_mean_within_auc', 'perm_p',
                 'perm_q_bh']], 20)
    show(wording, 20)

    # run record (with a checksum for every output file of this mode)
    outputs = []
    for p in sorted(q for q in out.rglob('*') if q.is_file()):
        if p.name == 'run_manifest.json':
            continue
        n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
        outputs.append({'file': p.relative_to(out).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
    cache_files = sorted(q for q in CACHE.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv'))
    manifest = {
        'settings_key': key, 'notebook': '05_image_model_check.ipynb', 'mode': mode,
        'labels': ('real' if shuffle_seed is None else 'shuffled (rehearsal)' if mode == 'rehearsal'
                   else 'shuffled (local code test)'), 'run_started_utc': RUN_STARTED,
        'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'in_colab': IN_COLAB, 'local_test_settings': TEST,
        'input_runs': {'01': man01['run_started_utc'], '02': man02['run_started_utc'], '03': man03['run_started_utc'],
                       '04': man04['run_started_utc']},
        'tasks': TASKS, 'reproduced_notebook03': REPRODUCED, 'same_versions_as_notebook03': same_versions,
        'patients_total': int(len(PATIENTS)), 'patients_embedded': int(len(complete)),
        'settings': {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM,
                     'MIN_COUNT': MIN_COUNT, 'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'LR_PARAMS': LR_PARAMS,
                     'BODY_HU': BODY_HU, 'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU,
                     'CTFM': [CTFM_REPO, CTFM_REVISION, CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP],
                     'MERLIN': [MERLIN_REPO, MERLIN_REVISION, MERLIN_CKPT], 'SWAP_MIN_DIFF_MM': SWAP_MIN_DIFF_MM,
                     'SWAP_MIN_GAP_MM': SWAP_MIN_GAP_MM, 'SWAP_SHIFT_MIN': SWAP_SHIFT_MIN,
                     'CONTROL_SIGMAS_MM': CONTROL_SIGMAS_MM, 'HDR_COLS': HDR_COLS, 'PREP_KEY': PREP_KEY,
                     'EMB_KEY': EMB_KEY},
        'versions': VERSIONS,
        'merlin_official_check': json.loads(merlin_check_file.read_text()) if merlin_check_file.exists() else None,
        'cache_files': {'count': len(cache_files),
                        'combined_md5': hashlib.md5(''.join(md5(q) for q in cache_files).encode()).hexdigest()},
        'outputs': outputs,
    }
    (out / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return probe, within, swap

In [ ]:
# 16. Run: rehearsal (shuffled labels) -> automatic gate -> real labels
modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results:
        probe_r, within_r, _ = results['rehearsal']
        null_means = pd.concat([probe_r.image_null_mean_auc, probe_r.background_null_mean_auc])
        problems = []
        if len(probe_r) != 2 * len(TASKS):
            problems.append('not every task and encoder finished')
        if not null_means.between(*GATE_NULL_RANGE).all():
            problems.append(f'a null mean AUC is outside {GATE_NULL_RANGE}: {null_means.round(3).tolist()}')
        if (cov['mean'] < GATE_MIN_COVERAGE).any():
            problems.append('too few scans embedded for a task')
        if problems:
            print('\nGATE NOT PASSED; the real-label pass was NOT run:', *problems, sep='\n - ')
            print('Inspect the run log.')
            break
        print('\nRehearsal gate passed: all analyses ran, null means look like chance, enough scans.')
    results[mode] = run_mode(mode)

In [ ]:
# 17. Plain-English summary, read back from the saved files
for mode in results:
    out = OUT_ROOT / mode
    pr = pd.read_csv(out / 'image_probe.csv')
    wd = pd.read_csv(out / 'shortcut_wording.csv')
    sw = pd.read_csv(out / 'image_swap.csv')
    lab_kind = json.loads((out / 'run_manifest.json').read_text())['labels']
    print(f"\n{mode.upper()} ({lab_kind}{': every number should look like chance' if lab_kind.startswith('shuffled') else ''})")
    for enc, g in pr.groupby('encoder'):
        print(f"- {enc}: image probe flagged in {int(g.image_flagged.sum())} of {len(g)} tasks; "
              f"background-only flagged in {int(g.background_flagged.sum())}; "
              f"null means {g.image_null_mean_auc.min():.3f}-{g.image_null_mean_auc.max():.3f}.")
    print(f"- Acquisition swap applicable in {int(sw.applicable.sum() / len(ENCODERS))} of {len(TASKS)} tasks; "
          f"rule met in {int(sw.rule_met.sum())} task-encoder pairs.")
    print(f"- 'Shortcut' wording allowed (plan 7d.7) for {int(wd.shortcut_wording_allowed.sum())} task-encoder pairs.")
print('\nNext: review the saved results and run log before writing up findings.')